# External SAELens activation experiments

This companion audits the completed AG News / DBpedia run: 3,300 documents,
four SAEs, two pooling operators, two source sizes, and thirteen methods.
GPU extraction was executed separately using the same package. Running this
notebook recomputes the score audit; it does not rerun backbone inference.

The fixed protocol, source identities and diagnostics accompany the cache.
Do not interpret bootstrap source-draw intervals as new-corpus uncertainty.
The best-single baseline outperforms the graded rule in every condition;
all methods, including unfavorable comparisons, remain in the report.

In [1]:
from pathlib import Path
import json
import os
import sys

# Run from a repository checkout, or set LATTICE_REPOSITORY explicitly.
repo = Path(os.environ.get("LATTICE_REPOSITORY", Path.cwd())).resolve()
while not (repo / "src/lattmc").exists() and repo != repo.parent:
    repo = repo.parent
if not (repo / "src/lattmc").exists():
    raise RuntimeError("Set LATTICE_REPOSITORY to the repository root.")
sys.path.insert(0, str(repo / "src"))
folder = repo / "notebooks/sae/data/external_results"
output = Path(os.environ.get("ACTIVATION_STUDY_OUTPUT", folder))
protocol = output / "external_protocol.md"
print("Cache:", output)
print("Interpreter:", sys.executable)

Cache: /Users/ltsinadze/git/my_papers/texs/sparsesurrs/lattcontexts/experiments/external_results
Interpreter: /Users/ltsinadze/git/my_papers/.venv/bin/python


## Independent metric and provenance audit

AP is independently recomputed from distinct score groups. The audit also
checks calibrated confusion counts, training-source isolation, source and
array hashes, and reported aggregate means. It needs the derived activation
archive as well as the smaller result archive, but no raw text or weights.

In [2]:
from lattmc.activationstudy.audit_codexgen import audit

result = audit(output, protocol)
print(json.dumps(result, indent=2))

{
  "query_configurations": 2880,
  "metric_vectors": 37440,
  "test_score_comparisons": 23712000,
  "status": "passed",
  "protocol_sha256": "4c34559cdf4a7e245f2091dff1198aa307885e5e51ea7b882575905f8a22ea48"
}


## All primary comparisons

Maximum pooling and three positive examples. Values are macro test AP in
percent, averaging all categories and ten source draws. Probes also use the
matched negative examples. Every method receives the same calibration
labels. The reported paired intervals are conditional on the fixed corpus.

In [3]:
import pandas as pd

rows = []
for path in sorted(output.glob("*/*_max_results.json")):
    report = json.loads(path.read_text())
    summary = report["summary"]["3"]
    row = {"corpus": report["dataset"], "SAE": report["checkpoint"]}
    for method in ("graded", "support", "best_single", "sae_probe",
                   "dense_probe", "tfidf_probe"):
        row[method] = 100 * summary["methods"][method]["mean"]
    rows.append(row)
pd.DataFrame(rows).round(2)

,corpus,SAE,graded,support,best_single,sae_probe,dense_probe,tfidf_probe
0,ag_news,gemma2_l0_301,29.94,27.79,65.02,62.96,73.35,39.59
1,ag_news,gemma2_l0_37,29.55,28.40,59.79,61.07,73.35,39.59
2,ag_news,gpt2_mlp8,31.58,29.27,49.95,51.89,57.18,39.59
3,ag_news,gpt2_res8,33.05,31.03,62.58,50.65,66.75,39.59
4,dbpedia_14,gemma2_l0_301,29.09,21.86,75.87,68.97,76.55,66.02
5,dbpedia_14,gemma2_l0_37,43.84,32.62,83.62,70.02,76.55,66.02
6,dbpedia_14,gpt2_mlp8,44.68,32.05,67.90,65.07,67.16,66.02
7,dbpedia_14,gpt2_res8,47.13,33.21,76.32,66.04,59.27,66.02


In [4]:
reports = [json.loads(path.read_text())
           for path in sorted(output.glob("*/*_results.json"))]
for shot in ("3", "10"):
    count = sum(r["summary"][shot]["full_empty"] for r in reports)
    print(shot, "positive sources:", count, "/ 1440 full extents empty")
assert all(s["methods"]["best_single"]["mean"]
           > s["methods"]["graded"]["mean"]
           for report in reports for s in report["summary"].values())
print("Best-single AP exceeds graded AP in all 32 conditions.")

3 positive sources: 1440 / 1440 full extents empty
10 positive sources: 727 / 1440 full extents empty
Best-single AP exceeds graded AP in all 32 conditions.


## Fresh inference and reconstruction

Use the existing project uv environment and the package README under
`src/lattmc/activationstudy/`. The commands there download pinned datasets,
select/deduplicate rows, tokenize them, extract GPT-2 and Gemma activations
sequentially, evaluate the frozen protocol, and audit outputs. Default
extraction is offline and uses the recorded Hugging Face cache revisions.

GPT-2 residual SAEs read **before** block 8; its MLP SAE reconstructs the
MLP output. Gemma SAEs read **after** block 8. Loading from the SAELens
registry is essential for the MLP SAE's normalization factor. The Gemma
implementation uses a Hugging Face hook followed by `SAE.encode`.

Raw benchmark text, token arrays, and third-party model weights are not in
the release candidates. Reconstruct them from the pinned upstream sources.
The prepared archives are local files, not evidence of a public deposit.

## Query type and token-witness presentation

This initial phase uses pooled document
meets, with maximum and mean pooling reported separately. Mean pooling
is not a coordinatewise join. The measured token gallery in
`context_depth_codexgen.ipynb` belongs to the later four-family phase;
it does not supply missing initial-phase GPT-2 or Gemma JumpReLU
counterparts. Do not transfer its highlights between checkpoints.